# CLIP features for the face crops

Turns every face crop into 512 numbers with a frozen CLIP image encoder (ViT-B/16, OpenAI weights).
One feature file per split, plus the matching manifest rows in the same order, so row *i* of the
features always belongs to row *i* of the table.

**Inputs (Add Input):** the crops dataset `NIC_Faces_FullDataset(ffpp_celebdf)`.
**Settings:** Accelerator GPU, Internet on (for pip and the CLIP weights).

**Test run:** keep `LIMIT_VIDEOS = 10` and run all cells. Expected check-cell output (from the
23 Oct 2026 test with the same settings):

| split | shape | mean norm |
|---|---|---|
| ffpp_train | (128, 512) | 10.31 |
| ffpp_val | (320, 512) | 10.71 |
| ffpp_test | (320, 512) | 10.68 |
| celebdf_test | (320, 512) | 10.86 |

**Full run:** set `LIMIT_VIDEOS = 0`, then Save Version, Save & Run All.

## 1. Settings

In [1]:
LIMIT_VIDEOS = 0
MODEL_NAME = "ViT-B-16-quickgelu"   # OpenAI's weights were trained with QuickGELU; plain "ViT-B-16" gives slightly wrong features
PRETRAINED = "openai"
BATCH_SIZE = 256
WORKERS = 8                # parallel image readers; reading the crops is the slow part, not the GPU
OUT_DIR = "/kaggle/working/features"
MACHINE_NAME = "kaggle-malak"
PARTS = ["ffpp/train", "ffpp/val", "ffpp/test", "celebdf/test"]

## 2. Install and imports

In [2]:
!pip install -q open_clip_torch==3.3.0

In [3]:
import os
import csv
import glob
import json
import time
import platform
from importlib.metadata import version

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import open_clip

## 3. Find the crops

Kaggle unpacks the crop archives into folders. The search goes level by level, so it stops
before it reaches the 107,000 image files (a full recursive search takes many minutes).

In [4]:
def find(pattern):
    for depth in range(0, 8):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if len(hits) > 0:
            return hits[0]
    raise FileNotFoundError(pattern)


inputs = {}
for part in PARTS:
    name = part.replace("/", "_")
    manifest = find(name + "_manifest.csv")
    root = find(part)[:-len(part)]
    inputs[name] = (manifest, root)
    print(name, manifest, root, sep="\n  ")

ffpp_train
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/ffpp_train_manifest.csv
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/ffpp_train/
ffpp_val
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/ffpp_val_manifest.csv
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/ffpp_val/
ffpp_test
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/ffpp_test_manifest.csv
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/ffpp_test/
celebdf_test
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/celebdf_test_manifest.csv
  /kaggle/input/datasets/malakkhaled202300369/nic-faces-fulldatasetffpp-celebdf/faces/celebdf_test/


## 4. Reading the crops

In [5]:
class CropDataset(torch.utils.data.Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.preprocess(img)


def read_manifest(path, limit_videos):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    if limit_videos > 0:
        # keep the first N videos, in manifest order, for a quick test
        keep = []
        seen = set()
        for row in rows:
            key = (row["method"], row["video_id"])
            if key not in seen:
                if len(seen) == limit_videos:
                    break
                seen.add(key)
            keep.append(row)
        rows = keep
    return rows

## 5. Load CLIP (frozen, never trained)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(device).eval()
print("device", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

device cuda Tesla T4


## 6. Extract features, one split at a time

Raw image embeddings are saved, not normalized; normalization is decided later, in one place.
Each split also appends one line to `runs.jsonl` with the settings, versions and time.

In [7]:
def extract(name, manifest, root):
    rows = read_manifest(manifest, LIMIT_VIDEOS)
    paths = [os.path.join(root, row["path"]) for row in rows]
    missing = [p for p in paths if not os.path.exists(p)]
    if len(missing) > 0:
        raise FileNotFoundError("%d of %d images missing, first: %s" % (len(missing), len(paths), missing[0]))

    loader = torch.utils.data.DataLoader(CropDataset(paths, preprocess), batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=WORKERS)
    start = time.time()
    chunks = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=name):
            batch = batch.to(device)
            if device == "cuda":
                with torch.autocast("cuda", dtype=torch.float16):
                    emb = model.encode_image(batch)
            else:
                emb = model.encode_image(batch)
            chunks.append(emb.float().cpu().numpy())
    features = np.concatenate(chunks, axis=0)
    seconds = time.time() - start

    np.save(os.path.join(OUT_DIR, name + "_clip.npy"), features)
    with open(os.path.join(OUT_DIR, name + "_rows.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    info = {
        "finished": time.strftime("%Y-%m-%d %H:%M:%S"),
        "name": name,
        "machine": MACHINE_NAME,
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "",
        "seconds": round(seconds, 1),
        "crops": int(features.shape[0]),
        "dim": int(features.shape[1]),
        "model": MODEL_NAME,
        "pretrained": PRETRAINED,
        "limit_videos": LIMIT_VIDEOS,
        "batch_size": BATCH_SIZE,
        "workers": WORKERS,
        "manifest": manifest,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "open_clip": version("open_clip_torch"),
    }
    with open(os.path.join(OUT_DIR, "runs.jsonl"), "a") as f:
        f.write(json.dumps(info) + "\n")
    print("%s: %d crops, %.1f min" % (name, features.shape[0], seconds / 60))


os.makedirs(OUT_DIR, exist_ok=True)
for name in inputs:
    manifest, root = inputs[name]
    extract(name, manifest, root)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
ffpp_train:   0%|          | 0/180 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ratio

ffpp_train: 46076 crops, 6.2 min


ffpp_val: 100%|██████████| 88/88 [03:02<00:00,  2.07s/it]


ffpp_val: 22396 crops, 3.0 min


ffpp_test: 100%|██████████| 88/88 [03:12<00:00,  2.18s/it]


ffpp_test: 22400 crops, 3.2 min


celebdf_test: 100%|██████████| 65/65 [02:17<00:00,  2.12s/it]


celebdf_test: 16572 crops, 2.3 min


## 7. Check

Every split should show 512 columns, the same number of rows as features, no NaN values,
and a mean norm close to the other splits. For the 10-video test, compare with the table at the top.

In [8]:
for name in inputs:
    x = np.load(os.path.join(OUT_DIR, name + "_clip.npy"))
    with open(os.path.join(OUT_DIR, name + "_rows.csv")) as f:
        n_rows = len(list(csv.DictReader(f)))
    print(name, x.shape, "rows", n_rows, "nan:", bool(np.isnan(x).any()),
          "mean norm: %.2f" % np.linalg.norm(x, axis=1).mean())

ffpp_train (46076, 512) rows 46076 nan: False mean norm: 10.49
ffpp_val (22396, 512) rows 22396 nan: False mean norm: 10.48
ffpp_test (22400, 512) rows 22400 nan: False mean norm: 10.55
celebdf_test (16572, 512) rows 16572 nan: False mean norm: 10.81
